<a href="https://colab.research.google.com/github/myazzeh/NLP-Course/blob/main/NLP_RNN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [7]:
from datasets import load_dataset
data = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [15]:
data["train"]['label']

Column([0, 0, 0, 0, 0])

In [21]:
x_train = list(data['train']['text'])
y_train= list(data['train']['label'])
x_test = list(data['test']['text'])
y_test= list(data['test']['label'])


In [32]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
tok= Tokenizer()
tok.fit_on_texts(x_train)
train_sequences = tok.texts_to_sequences(x_train)
test_sequences = tok.texts_to_sequences(x_test)
vocab = tok.word_index

In [30]:
print(test_sequences[0:4])
print(len(test_sequences[0]))
print(len(test_sequences[1]))

[[10, 116, 918, 923, 2, 241, 1687, 5, 273, 53, 16, 3, 173, 918, 923, 99, 245, 23, 629, 48020, 463, 2524, 2, 7220, 10, 800, 5, 37, 11, 10, 63, 119, 18, 9, 6, 5, 49, 245, 918, 923, 14, 15162, 454, 6, 5, 320, 2140, 1, 201, 705, 18962, 703, 3444, 729, 4572, 3958, 4428, 12, 149, 1012, 1, 972, 2, 2143, 28, 2039, 102, 562, 27, 3080, 16, 3, 60970, 60971, 954, 143, 249, 47, 23, 145, 4, 22, 43, 47, 34, 101, 15162, 454, 6, 49, 918, 923, 245, 42, 21, 42, 2708, 2, 8656, 134, 175, 795, 235, 37, 1422, 2, 106, 940, 918, 923, 6, 3, 509, 12, 124, 21, 190, 407, 612, 21084, 320, 2140, 9, 200, 1690, 671, 1335, 243, 21, 14, 3, 618, 4029, 42, 63, 876, 5, 456, 41, 1, 102, 130, 14, 33, 23, 21, 328, 6415, 40, 1010, 3, 5367, 4, 110, 65, 1726, 2, 3388, 23, 1638, 2, 724, 397, 1348, 5, 103, 1, 1185, 4, 700, 121, 42, 1912, 14, 33, 25, 5, 207, 132, 1925, 34830, 700, 896, 81, 59, 21, 1742, 146, 34830, 11588, 212, 27, 1584, 8, 65, 15734, 14, 11, 750, 703, 858, 1987, 146, 9, 206, 16666, 2012, 63, 958, 11, 341, 35880, 4,

In [35]:
L=200
d=300
h=100

train_sequences= pad_sequences(train_sequences, maxlen=L, padding='post', truncating='post')
test_sequences= pad_sequences(test_sequences, maxlen=L, padding='post', truncating='post')

In [37]:
len(train_sequences[100])

200

In [40]:
from tensorflow.keras.layers import InputLayer, SimpleRNN, Dense, Flatten, Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.models import Sequential

In [46]:
len(vocab)

88582

In [43]:
v=len(vocab)
NN = Sequential ([
    InputLayer(shape= (L,)),
    Embedding(v, d),
    SimpleRNN(h, return_sequences= True),
    SimpleRNN(h, return_sequences= False),
    Dense(50),
    Dense(25),
    Dense(1, activation="sigmoid")
    ])
NN.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 300)       │    26,574,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 200, 100)       │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 100)            │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,641,151 (101.63 MB)

 Trainable params: 26,641,151 (101.63 MB)

 Non-trainable params: 0 (0.00 B)

In [45]:
import numpy as np

NN.compile(optimizer='Adam', loss=BinaryCrossentropy(), metrics=['accuracy'])
NN.fit(train_sequences, np.array(y_train, dtype=np.float32), epochs=10)

Epoch 1/10
731/782 ━━━━━━━━━━━━━━━━━━━━ 25s 495ms/step - accuracy: 0.4914 - loss: 0.7095

InvalidArgumentError: Graph execution error:

Detected at node sequential_1/embedding_1_1/GatherV2 defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.1

In [ ]:
NN.evaluate(test_sequences, np.array(y_test, dtype=np.float32))

In [ ]:
#Inferense

My_comment=" I hate that movie"

seq= tok.texts_to_sequences(My_comment)
seq= pad_sequences(seq, maxlen=L, padding='post', truncating='post')
y_pred = NN(seq)
# display y_pred on the front end